In [75]:
import polars as pl
import quak
import traitlets
import duckdb

import boro

In [76]:
class RowCounter(boro.Client):
    _esm = """
    export default {
      async render({ model, host, signal, el }) {
        el.style.cssText = "font: 14px ui-sans-serif; padding: 6px 8px;";
        el.textContent = "…";
        const coord = await host.getWidget(model.get("coord"));
        const { msql, createClient } = coord.exports;
        const ctx = await createClient({ model, host, signal });
        const rows = ctx.query(
          (filter) =>
            msql.Query.from(model.get("table"))
              .select({ n: msql.count() })
              .where(filter),
          { filterBy: ctx.filterBy },
        );
        rows.addEventListener("value", (result) => {
          if (result.isError) {
            el.textContent = `error: ${result.error.message}`;
            return;
          }
          if (!result.isSuccess) return;
          const n = Number(result.data.toColumns().n[0]);
          el.textContent = `${n.toLocaleString()} rows`;
        }, { signal });
      },
    };
    """
    table = traitlets.Unicode().tag(sync=True)

    def __init__(self, coord: boro.Coordinator, table: str, **kwargs: object) -> None:
        super().__init__(coord=coord, table=table, **kwargs)


In [77]:
con = duckdb.connect()
href = "https://github.com/uwdata/mosaic/raw/main/data/athletes.parquet"
href = "/Users/tmanz/demos/boro-vgplot/data/flights-10m.parquet"
href = "./data/wnba-shots-2023.parquet"
con.execute(f"CREATE TABLE df AS SELECT * FROM read_parquet('{href}')")
con.sql("DESCRIBE df")

┌──────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name  │ column_type │  null   │   key   │ default │  extra  │
│   varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ type         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ description  │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ score_value  │ TINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ x_position   │ TINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ y_position   │ TINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ season_type  │ TINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ game_date    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ game_id      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ qtr          │ TINYINT     │ YES     │ NULL    │ NULL    │ NULL    │
│ athlete_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ team

In [88]:
coord = boro.Coordinator.connect(con);
sel = boro.Selection.crossfilter(coord);

dt = quak.DataTable(coord, table = "df", selection = sel);
dt;

In [89]:
RowCounter(coord, table = "df", selection = sel);

In [90]:
dt.col("weight").set([20, 100]);
dt.col("sport").set("athletics");
dt.col("sex").set("male");

In [91]:
dt.col("weight");

ColumnHandle('weight', kind=None, value=[20, 100])

In [92]:
dt.sql;

'SELECT * FROM "df"'

In [93]:
from examples.court_scatter import CourtScatter

court = CourtScatter(
    coord, table="df", x="x_position", y="y_position",
    color="type",              # categorical column → legend + point colors
    selection=(sel, None),     # ← filter-only: reads sel, never writes to it
)
court